In [ ]:
"""
OWF supply chain analysis computational model

Article: Targeting Made-in-Europe rules in wind energy auctions: Which components should be covered?
Authors: Raúl Labarthe, Ulrich Elmer Hansen, and Anastasia Ioannou

Main reproducibility notebook for the paper model. All paths are defined in
modules.config as the single source of truth for the local run tree.
"""

from __future__ import annotations
import importlib
from pathlib import Path

import modules.config as config

config = importlib.reload(config)
config.verify_requirements()

import numpy as np
import pandas as pd

PAPER_TITLE = "Targeting Made-in-Europe rules in wind energy auctions: Which components should be covered?"
PAPER_AUTHORS = "Raúl Labarthe, Ulrich Elmer Hansen, and Anastasia Ioannou"
MODEL_VERSION = "paper-submission"

# ============================================================================
# CENTRALIZED CONFIGURATION
# ============================================================================

from modules.config import (
    # Root directories
    ROOT,
    PROJECT_ROOT,
    CALC_ROOT,
    MODULES_ROOT,
    # Generated output paths
    RESULTS_IMAGES,
    RESULTS_TABLES,
    RESULTS_SUPPLEMENTARY,
    RESULTS_CACHE,
    IMG_ROOT,
    TAB_ROOT,
    # Cost model paths
    COST_MODEL_ROOT,
    COST_SOURCES,
    EXCHANGE_ROOT,
    # Supply chain paths
    SUPPLY_CHAIN_ROOT,
    SUPPLY_SOURCES,
    TURBINE_DATA_ROOT,
    WINDFARM_DATA_ROOT,
    # MRIO analysis paths
    MRIO_ROOT,
    DATASETS_DIR,
    HELPERS_DIR,
    METADATA_DIR,
    MRIO_SCENARIOS_DIR,
    # Results
    project_relative_path,
)

PAPER_ROOT = PROJECT_ROOT


In [ ]:
# ============================================================================
# GLOBAL CONFIGURATION FLAGS
# ============================================================================

CLOSE_FIGURES = True
SAVE_FIGURES = True
SAVE_MAPS = True
SAVE_FIGURES_RES = True
RECOMPUTE = True
FIRST_MRIO = True
FIRST_EMP = True
YEAR = 2024
YEAR_MIN = 2024

HOUR_DATAFLOWS = (
    "DF_HOW_TEMP_SEX_EC2_NB",
    "DF_HOW_TEMP_SEX_ECO_NB",
)
EMP_DATAFLOWS = (
    "DF_EMP_2EMP_SEX_ECO_NB",
    "DF_EMP_TEMP_SEX_EC2_NB",
    "DF_EMP_TEMP_SEX_ECO_NB",
)
FTE40_DATAFLOW = "DF_EMP_2FTE_SEX_JBF_NB"
FTE40_BASIS = "JBF_FTE_FTE40"
WORLD_EMPLOYMENT_DATAFLOW = "DF_EMP_2EMP_SEX_ECO_NB"

# ============================================================================
# CONSTANTS
# ============================================================================

EU27 = {"AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", 
        "FR", "GR", "HR", "HU", "IE", "IT", "LT", "LU", "LV", "MT", 
        "NL", "PL", "PT", "RO", "SE", "SI", "SK"} 

# EFTA members: Switzerland, Iceland, Liechtenstein, and Norway
EFTA = {"CH", "IS", "LI", "NO"}
EU_EFTA = EU27 | EFTA
EU_EFTA_UK = EU_EFTA | {"GB"}

# Additional European countries represented in the current FIGARO release
EUROPE_EXTRA = {"AL", "ME", "MK", "RS", "RU", "TR"}
EUROPE = EU_EFTA_UK | EUROPE_EXTRA
EUROPE_GROUP = EUROPE

CORE_DEPLOYMENT = ["GB", "BE", "NL", "DK", "DE"]
EU27_DEP = [d for d in CORE_DEPLOYMENT if d in EU27]
EU27_AVG = "EU27_AVG"

In [ ]:
# Verify imports are loaded and show configuration summary
print("=" * 70)
print("✓ CONFIGURATION LOADED FROM modules.config")
print("=" * 70)
print("\n📁 ROOT DIRECTORIES:")
print(f"   ROOT: {project_relative_path(ROOT)}")
print(f"   PROJECT_ROOT: {project_relative_path(PROJECT_ROOT)}")
print(f"   CALC_ROOT: {project_relative_path(CALC_ROOT)}")
print(f"   MODULES_ROOT: {project_relative_path(MODULES_ROOT)}")

print("\n📊 COST MODEL PATHS:")
print(f"   COST_MODEL_ROOT: {project_relative_path(COST_MODEL_ROOT)}")
print(f"   COST_SOURCES: {project_relative_path(COST_SOURCES)}")
print(f"   EXCHANGE_ROOT: {project_relative_path(EXCHANGE_ROOT)}")

print("\n📈 SUPPLY CHAIN PATHS:")
print(f"   SUPPLY_CHAIN_ROOT: {project_relative_path(SUPPLY_CHAIN_ROOT)}")
print(f"   SUPPLY_SOURCES: {project_relative_path(SUPPLY_SOURCES)}")
print(f"   TURBINE_DATA_ROOT: {project_relative_path(TURBINE_DATA_ROOT)}")
print(f"   WINDFARM_DATA_ROOT: {project_relative_path(WINDFARM_DATA_ROOT)}")

print("\n📋 MRIO ANALYSIS PATHS:")
print(f"   MRIO_ROOT: {project_relative_path(MRIO_ROOT)}")
print(f"   DATASETS_DIR: {project_relative_path(DATASETS_DIR)}")
print(f"   HELPERS_DIR: {project_relative_path(HELPERS_DIR)}")
print(f"   METADATA_DIR: {project_relative_path(METADATA_DIR)}")
print(f"   MRIO_SCENARIOS_DIR: {project_relative_path(MRIO_SCENARIOS_DIR)}")

print("\n✓ COST MODEL MODULES IMPORTED:")
print("   - inputs, helpers, model, charts")

print("\n⚙️ ANALYSIS PARAMETERS:")
print(f"   - Save figures: {SAVE_FIGURES}")
print(f"   - Save figures results: {SAVE_FIGURES_RES}")
print(f"   - Close figures: {CLOSE_FIGURES}")
print(f"   - Year: {YEAR}")
print(f"   - Core deployment: {CORE_DEPLOYMENT}")
print("=" * 70)


### Cost Model

In [ ]:
# ============================================================================
# Cost Model Module 
# ============================================================================

import importlib
import sys
from dataclasses import dataclass

# Reload modules if already loaded
for module_name in ['modules.cost_model', 'modules.cost_model.inputs',
                    'modules.cost_model.model', 
                    'modules.cost_model.charts']:
    if module_name in sys.modules:
        try:
            importlib.reload(sys.modules[module_name])
        except:
            pass


from modules.cost_model.inputs import (
    build_fx_dictionary,
    PlantConfig,
    costs_bvg,
    NREL_ROWS,
)

from modules.cost_model.model import (
    turbine_masses,
    build_turbine_supply,
    build_cost_breakdown,
)

from modules.cost_model.charts import (
    plot_turbine_double_ring,
    plot_model_double_donut,
)

print("✓ All cost_model imported successfully")


Exchange rates


In [ ]:
# Build FX dictionary from exchange rates
# Returns empty dict if no data available (optional component)
fx_dict = build_fx_dictionary(EXCHANGE_ROOT)
print(f"✓ FX dictionary loaded with {len(fx_dict)} currencies" if fx_dict else "⚠ No FX data available (optional)")

### Turbines


Exchange rates


In [ ]:
# Exchange rates
eur_gbp = 1/fx_dict['GBP'][2024]
eur_usd= 1/fx_dict['USD'][2024] 


Turbine design


In [ ]:
@dataclass
class TurbineDesign:
    """
    European Offshore Wind Farm turbine design.

    Default values based on BVG (2025) and DTU 10MW reference turbine.
    
    """
    n_turbines: int = round(1000 / 10)  # 100 turbines for 1 GW farm
    rated_rpm: float = 9.6
    rotor_diameter: float = 178.3
    rated_power: float = 10  # MW
    hub_height: float = 119  # m
    drivetrain_type: str = "dd"  # 'dd' (direct drive), 'hs' (high-speed), 'ms' (medium-speed)
    foundation_type: str = "monopile"


Turbine cost

In [ ]:
turbine_supply_cost = build_turbine_supply(
    TurbineDesign(),
    turbine_masses(TurbineDesign()),
    eur_gbp=eur_gbp,          
    calibrate_to_bvg=True,
)


Lifetime cost breakdown


In [ ]:
# Analysis parameters
lifetime_years = 25

cost_breakdown = build_cost_breakdown(
        turbine_supply_cost,
        PlantConfig(),
        TurbineDesign(),
        project_duration_years=lifetime_years,
)

Plots

In [ ]:
# A.2 and A.3
plot_model_double_donut(
    cost_breakdown,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

plot_turbine_double_ring(
    turbine_supply_cost,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

In [ ]:
# Run the cost model for each country
if __name__ == "__main__":

    # Dictionary that will store results for all countries
    cost_by_country = {}

    # ---- Loop through all countries ----
    for country_deployment in CORE_DEPLOYMENT:
        print(f"▶ Running model for country: {country_deployment}")
        # Build the cost breakdown for this country
        cost_breakdown_n = build_cost_breakdown(
            turbine_supply_cost,
            PlantConfig(),
            TurbineDesign(),
            project_duration_years=lifetime_years,
        )
        cost_breakdown_n.pop("ABEX", None)
        # Store the results
        cost_by_country[country_deployment] = cost_breakdown_n

### Supply chain analysis


Turbine components procurement - WindEurope


In [ ]:
# ============================================================================
# Supply Chain Module 
# ============================================================================

import importlib
import sys

for module_name in ['modules.supply_chain.inputs',
                    'modules.supply_chain.turbine_analysis', 
                    'modules.supply_chain.suppliers_analysis',
                    'modules.supply_chain.charts']:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from modules.supply_chain.inputs import (
    import_shares,
    connected_country,
    power_by_country,
    CG_STD_MAP,
    import_category,
)

from modules.supply_chain.turbine_analysis import (
    split_turbine_costs_by_oem,
    flatten_turbine_costs,
    add_category_to_df_turbine,
    allocate_all,
    build_turbine_trade,
    prepare_oem_share_by_country,
    oem_suppliers,
    prepare_bom_facilities,
    filter_bom_facilities_by_component_deploy,
    load_b2_turbine_inputs,
)

from modules.supply_chain.charts import (
    plot_alloc_share_stacked,
    plot_gb_local_benchmark,
    plot_macro_vertical,
    build_weighted_avg_flows_hier,
    plot_eu27_weighted_sourcing,
    freq_costtype_to_latex,
)

from modules.supply_chain.inputs import bridge, ownership_oem, oem_hq_country
from modules.country import COUNTRY_COORDS

from modules.supply_chain.suppliers_analysis import (
    build_trade_matrices_from_b1_sourcing,
    build_component_flows,
    build_component_flows_checked,
)

from openpyxl import load_workbook

print("✓ All supply_chain modules imported and data loaded successfully")


In [ ]:
# ============================================================================
# Load Supply Chain Data
# ============================================================================

# Public B2 entry point: input sheets from the supplementary workbook.
# The allocation process below is still executed from these inputs.
B2_SUPPLEMENTARY_WORKBOOK = TURBINE_DATA_ROOT / "Supplementary_Material_B2_Turbine_Supply_Chain_Sourcing_Inputs.xlsx"
if not B2_SUPPLEMENTARY_WORKBOOK.exists():
    raise FileNotFoundError(
        f"Missing public B2 supplementary workbook: {B2_SUPPLEMENTARY_WORKBOOK}"
    )

b2_inputs = load_b2_turbine_inputs(B2_SUPPLEMENTARY_WORKBOOK)
turbine_orders_raw = b2_inputs["oem_orders"]
share_dict, capacity_by_country, oem_share_df, turbine_orders = prepare_oem_share_by_country(turbine_orders_raw)

# Build allocations manually
cost_by_country_oem = split_turbine_costs_by_oem(cost_by_country, share_dict)
df_turbine = pd.DataFrame(flatten_turbine_costs(cost_by_country_oem))
df_turbine["component_group"] = df_turbine["component_group"].replace(CG_STD_MAP)
df_turbine_cat = add_category_to_df_turbine(df_turbine, bridge)

In [ ]:
# OEM-specific known suppliers from B2
oem_supply_raw = b2_inputs["known_suppliers"]
summary_table, oem_supply = oem_suppliers(oem_supply_raw, component_group_map=CG_STD_MAP)

# Assembly facilities from B2
assembly_fac = b2_inputs["assembly_facilities"]

# General manufacturing facilities from B2
bom_fac_raw = b2_inputs["general_facilities"]

KEEP = {"Castings", "Forgings", "Converters", "Tower"}

bom_keep = prepare_bom_facilities(
    bom_fac_raw,
    keep=KEEP,
    include_eu27=True,
    deploy=set(CORE_DEPLOYMENT),
)

bom_facilities = filter_bom_facilities_by_component_deploy(
    bom_keep,
    df_turbine_cat,
    share_thresh=0.1,
)


Cost allocation


In [ ]:
df_alloc = allocate_all(
    df_turbine_cat=df_turbine_cat,
    oem_supply=oem_supply,
    assembly_facilities=assembly_fac,
    bom_facilities=bom_facilities,
    ownership_oem=ownership_oem,
    oem_hq_country=oem_hq_country,
    country_coords=COUNTRY_COORDS,
)

turbine_trade = build_turbine_trade(df_alloc)


In [ ]:
# ==============================================================
# EU27 weighted turbine allocation
# ==============================================================

weights = {
    d: connected_country[d]["Power (MW)"]
    for d in EU27_DEP
}

total_weight = sum(weights.values())
weights = {d: w / total_weight for d, w in weights.items()}

In [ ]:
df_alloc_eu27 = df_alloc[
    df_alloc["deployment_country"].isin(EU27_DEP)
].copy()

df_alloc_eu27["value_alloc"] *= (
    df_alloc_eu27["deployment_country"].map(weights)
)

df_alloc_eu27["deployment_country"] = EU27_AVG



In [ ]:
plot_alloc_share_stacked(df_alloc_eu27, 
                         deployment_country=EU27_AVG, 
                         filename="B.2.2.pdf",
                         save=SAVE_FIGURES, 
                         close=CLOSE_FIGURES)

Other Wind Farm components - 4C Offshore


In [ ]:
# Public B1 entry point: processed sourcing_long from the supplementary workbook.
# The raw 4C Offshore pipeline is retained in modules.supply_chain.suppliers_analysis
# for licensed/internal regeneration, but it is not required for the public run.
B1_SUPPLEMENTARY_WORKBOOK = WINDFARM_DATA_ROOT / "Supplementary_Material_B1_Suppliers_Processing_Wind_Farm_4COffshore.xlsx"
CORE_ROW_ORDER = ["GB", "BE", "NL", "DK", "DE"]

if not B1_SUPPLEMENTARY_WORKBOOK.exists():
    raise FileNotFoundError(
        f"Missing public B1 supplementary workbook: {B1_SUPPLEMENTARY_WORKBOOK}"
    )

print(f"Using public B1 sourcing workbook: {project_relative_path(B1_SUPPLEMENTARY_WORKBOOK)}")


In [ ]:
# =====================================================================
# Final: flows + plots (clean pipeline)
# =====================================================================

# 1) Reviewed contract shares from public B1 sourcing_long (before local-content correction)
contract_shares_2000_2025 = build_trade_matrices_from_b1_sourcing(
    B1_SUPPLEMENTARY_WORKBOOK,
    df_alloc=df_alloc,
    period="2000-2025",
    share_version="total",
    core_row_order=CORE_ROW_ORDER,
)

# 2) Build flows BEFORE adjustment (for benchmark plot)
flows_raw, *_ = build_component_flows(
    cost_by_country=cost_by_country,
    trade_matrices=contract_shares_2000_2025,
    df_alloc=df_alloc,
)


In [ ]:
# 3) Final public sourcing shares from B1 sourcing_long.
# Uses share_total plus final_local_c to reconstruct normalized post-correction shares.
contract_shares_2000_2025_adj = build_trade_matrices_from_b1_sourcing(
    B1_SUPPLEMENTARY_WORKBOOK,
    df_alloc=df_alloc,
    period="2000-2025",
    share_version="final",
    core_row_order=CORE_ROW_ORDER,
)

# 4) Build flows AFTER adjustment + audit
flows_by_component_hier_all, flows_turbine_detail_hier, flows_turbine_L1, all_origins, missing_cols_df, colsum_issues_df = (
    build_component_flows_checked(
        cost_by_country=cost_by_country,
        trade_matrices=contract_shares_2000_2025_adj,
        df_alloc=df_alloc,
        required_deploy=["GB","BE","NL","DK","DE"],
        tol=1e-6,
    )
)

# Representative EU27 sourcing, weighted by connected offshore capacity.
flows_eu27_avg = build_weighted_avg_flows_hier(
    flows_by_component_hier_all,
    connected_country,
    avg_dep_order=EU27_DEP,
    label=EU27_AVG,
)

print("\n=== Missing/extra deployment columns per matrix ===")
print(missing_cols_df if len(missing_cols_df) else "None ✅")

print("\n=== Column-sum issues (should be 1.0) ===")
print(colsum_issues_df if len(colsum_issues_df) else "None ✅")


In [ ]:
# The paper run generates only the supplier-contract count table used in the manuscript.
wb = load_workbook(B1_SUPPLEMENTARY_WORKBOOK, read_only=True, data_only=True)
ws = wb["contracts_summary"]
rows = list(ws.iter_rows(values_only=True))
wb.close()

header = [str(x).strip() for x in rows[3] if x is not None]
data = [list(r[:len(header)]) for r in rows[4:] if r and r[0] is not None]
supplier_contracts_country = pd.DataFrame(data, columns=header)

country_cols = [c for c in ["BE", "DE", "DK", "GB", "NL", "Grand Total"] if c in supplier_contracts_country.columns]
for c in country_cols:
    supplier_contracts_country[c] = pd.to_numeric(supplier_contracts_country[c], errors="coerce").fillna(0).astype(int)

latex = freq_costtype_to_latex(
    supplier_contracts_country,
    caption="Supplier contracts by component, subcomponent, and deployment country.",
    label="tab:supplier_contracts_country",
    tabcolsep=2,
    arraystretch=1.0,
    font_size=r"\scriptsize",
)

supplier_contracts_path = Path(TAB_ROOT) / "supplier_contracts_country.tex"
supplier_contracts_path.write_text(latex, encoding="utf-8")
print(f"OK supplier contract table saved: {project_relative_path(supplier_contracts_path)}")

In [ ]:
# B.1.3
plot_gb_local_benchmark(
    flows_raw,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

Sourcing plot


In [ ]:
# 3.1
# Countries shown separately from the EU27 aggregate.
MACRO_SPLIT_COUNTRIES = ("DK", "NL", "DE", "BE","ES", "FR", "NO", "IT", "SE", "RO","GR","FI","PL", "AT", "CN","TR","AE","KR","US",)

plot_macro_vertical(
    flows_by_component_hier_all,
    split_iso=MACRO_SPLIT_COUNTRIES,
    filename="B.1.4.pdf",
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)


In [ ]:
EU27_SPLIT_COUNTRIES = ("DK", "NL", "DE", "BE","ES", "FR", "IT", "SE", "RO","GR","FI","PL", "AT","PT",)

plot_eu27_weighted_sourcing(
    flows_eu27_avg,
    split_iso=EU27_SPLIT_COUNTRIES,
    label_min_share=0.04,
    filename="3.1_EU_sourcing.pdf",
    save=SAVE_FIGURES_RES,
    close=CLOSE_FIGURES,
)

In [ ]:
ROW_SPLIT_COUNTRIES = ("GB","NO", "CN","TR","AE","KR","US","JP","SG","CH","RS","CA")

plot_eu27_weighted_sourcing(
    flows_eu27_avg,
    split_iso=ROW_SPLIT_COUNTRIES,
    label_min_share=0.04,
    filename="3.1_EU_sourcing_EU27.pdf",
    save=SAVE_FIGURES_RES,
    close=CLOSE_FIGURES,
)

### MRIO Construction


In [ ]:
# ============================================================================
# MRIO Construction Module 
# ============================================================================
import importlib
import sys
from typing import Literal

for module_name in ['modules.mrio_analysis.industrial',
                    'modules.mrio_analysis.charts',
                    'modules.mrio_analysis.employment', 
                    'modules.mrio_analysis.eurostat', 
                    'modules.mrio_analysis.ilostat', 
                    'modules.mrio_analysis.oecd',
                    'modules.mrio_analysis.employment_outputs',
                    'modules.mrio_analysis.mrio']:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from modules.mrio_analysis.industrial import (
    run_industrial,
    results_by_origin_industry_groups,
    plot_eu27_composition_groups,
)

from modules.config import EMPLOYMENT_DATA_DIR, HELPERS_DIR

from modules.mrio_analysis.mrio import (
    build_or_load_mrio,
    extract_gva_basic,
    calculate_A,
    calculate_C,
    calculate_vn_pj,
    calculate_L,
    _row_from_v,
    validate_mrio_coefficients,
    load_classifications,
    plot_lb_box_all_industries,
)

from modules.mrio_analysis.eurostat import (
    load_emp_eurostat_bundle,
)

from modules.mrio_analysis.ilostat import (
    show_preferred_dataflows,
    load_ilo_mean_hours_by_industry,
    load_ilo_employment_by_industry,
    load_ilo_global_employment,
    load_ilo_fte40,
)

from modules.mrio_analysis.employment_outputs import (
    build_fte40_source_decision_report,
    build_fte40_allocation_method_comparison,
    export_wage_supplementary_material,
    plot_wages_dispersion_boxplot_sorted,
)

from modules.mrio_analysis.employment import (
    assess_industrial_resolution_inventory,
    prepare_fte40_employment_inputs,
    build_fte40_country_industry,
    build_world_fte40_blocks,
    apply_world_a14_row_residual,
    load_fte40_results_cache,
)

from modules.mrio_analysis.oecd import (
    audit_oecd_c2_sources,
    apply_oecd_c2_production_overrides,
    OECD_C2_SELECTION_FINGERPRINT,
)

print("✓ All mrio_analysis modules imported and data loaded successfully")

In [ ]:
# ==============================================================
# RUN Industrial allocation
# ==============================================================
ind_ind = True

ind_codes, final_codes, primary_inputs, ind_labels = load_classifications(HELPERS_DIR, use_industry_matrix=ind_ind)

final_cost_df_agg, component_groups, subcomponents_dict = run_industrial(
    ROOT=str(HELPERS_DIR),
    flows_by_component_hier_all=flows_eu27_avg,
    df_alloc=df_alloc_eu27,
    ind_ind=ind_ind,  
)
tot_dep = final_cost_df_agg.groupby("deployment_country", as_index=False)["value_alloc"].sum()
tot_dep.sort_values("value_alloc", ascending=False)

In [ ]:
# C.1.1
plot_eu27_composition_groups(
    EU27_AVG,
    final_cost_df_agg,
    SAVE_FIGURES,
    CLOSE_FIGURES,
    fig_name="C.1.1",
    ind_ind=ind_ind,
)

Classification results


In [ ]:
# ==============================================================
# REGION INTO ANALYSIS
# ==============================================================

# Offshore Financial Centers
# https://www.imf.org/external/pubs/ft/wp/2007/wp0787.pdf
# https://www.ecb.europa.eu/press/economic-bulletin/articles/2020/html/ecb.ebart202002_01~1a58c02776.en.html?utm_source=chatgpt.com

CORE = { 
    # EU27
    "AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", "FR", "GR",
    "HR", "HU", "IT", "LT", "LV", "NL", "PL", "PT", "RO",
    "SE", "SI", "SK",
    # EU27 financial hubs
    "LU", "MT", "CH", "IE",
    # Relevant for Europe
    "GB", "NO", "TR", "RU", "IS", "RS",
    # Relevant trade partners
    "CN", "US", "JP", "CA", "KR", "IN", "AU","ZA","ID",
    # Additional countries present in the FIGARO release
    "AL", "ME", "MK",
    ## Less relevant trade partners ##
    # # ROW
    "SA",  
    # # South America
    "AR", "BR", "MX",
    # Value added components
    "W2"
}

In [ ]:
# Source: https://circabc.europa.eu/ui/group/cec66924-a924-4f91-a0ef-600a0531e3ba/library/15c722eb-702b-42e0-b1c6-00e7dd3c3198/details?download=true

MRIO = build_or_load_mrio(
    CORE=CORE,
    mrio_year=YEAR,
    use_cache=True,
    refresh_cache=FIRST_MRIO,
    save_raw_parquet=FIRST_MRIO,
    validate=True,
    use_industry_matrix=ind_ind,
    download_matrix=FIRST_MRIO,
)

# Use the regions actually represented in the prepared FIGARO matrix.
MRIO_REGIONS = (
    set(MRIO.index.get_level_values("origin_region").astype(str))
    - {"ROW", "W2"}
)
if len(MRIO_REGIONS) != 49:
    raise ValueError(f"Expected 49 explicit MRIO countries, found {len(MRIO_REGIONS)}.")

In [ ]:
# ==============================================================
# Macro
# ==============================================================
# Technical coefficients, final demand, primary inputs, and Leontief inverse
A_MRIO  = calculate_A(MRIO)
C_MRIO  = calculate_C(MRIO)
V_MRIO  = calculate_vn_pj(MRIO)
L1_MRIO = calculate_L(A_MRIO, V_MRIO, C_MRIO, system_type=1)  # Type I

# A64 industry classification
mrio_prd_set = set(L1_MRIO.columns.get_level_values("prd_use").astype(str))
A64_IND = mrio_prd_set

# Gross output by country-industry (million EUR)
# Excludes net taxes on products (D21X31), consistent with the
# gross-output definition used in the MRIO technical coefficients.
_row_codes = MRIO.index.get_level_values("prd_ava").astype(str)
X_MRIO = MRIO.loc[~_row_codes.isin(final_codes), L1_MRIO.columns].sum(axis=0)

# ==============================================================
# Factors of production
# ==============================================================
# Capital income coefficient (may be negative in some industries)
K_MRIO = _row_from_v(V_MRIO, "B2A3G").reindex(L1_MRIO.columns).fillna(0.0)
# Labour income coefficient (typically non-negative)
LB_MRIO = _row_from_v(V_MRIO, "D1").reindex(L1_MRIO.columns).fillna(0.0)
# Other taxes less subsidies on production
T_MRIO = _row_from_v(V_MRIO, "D29X39").reindex(L1_MRIO.columns).fillna(0.0)
# Net taxes on products
TP_MRIO = _row_from_v(V_MRIO, "D21X31").reindex(L1_MRIO.columns).fillna(0.0)
# Factor-income capture coefficient (labour + capital)
I_MRIO = LB_MRIO + K_MRIO
# Value added including net taxes on products (TiVA convention)
VA_MRIO = LB_MRIO + K_MRIO + T_MRIO + TP_MRIO
# Value added in basic prices (excludes net taxes on products)
VA_BASIC_MRIO = extract_gva_basic(MRIO).reindex(L1_MRIO.columns)
# validate_mrio_coefficients(A_MRIO, V_MRIO, VA_MRIO)

EUROSTAT


In [ ]:
# Final FTE40 cache: source inputs are never loaded when this is valid.
_employment_cache_pairs = tuple(
    (str(geo), str(industry)) for geo, industry in L1_MRIO.columns
)
EMPLOYMENT_CACHE_METADATA = {
    "schema": "fte40_country_industry_v1",
    "year": int(YEAR),
    "year_min": int(YEAR_MIN),
    "industry_matrix": bool(ind_ind),
    "mrio_regions": tuple(sorted(map(str, MRIO_REGIONS))),
    "mrio_industries": tuple(sorted(map(str, A64_IND))),
    "mrio_columns": _employment_cache_pairs,
    "hour_dataflows": tuple(HOUR_DATAFLOWS),
    "employment_dataflows": tuple(EMP_DATAFLOWS),
    "fte40_dataflow": FTE40_DATAFLOW,
    "fte40_basis": FTE40_BASIS,
    "world_employment_dataflow": WORLD_EMPLOYMENT_DATAFLOW,
    "world_row_method": "world_a14_residual_output_relative_intensity_v1",
    "oecd_c2_selection_fingerprint": OECD_C2_SELECTION_FINGERPRINT,
}
EMPLOYMENT_CACHE_PATH = RESULTS_CACHE / f"fte40_{'ind_by_ind' if ind_ind else 'prd_by_prd'}.pkl"
EMPLOYMENT_AUDIT_CACHE_PATH = RESULTS_CACHE / f"fte40_source_audits_{'ind_by_ind' if ind_ind else 'prd_by_prd'}.pkl"
_cached_fte = None if FIRST_EMP else load_fte40_results_cache(
    EMPLOYMENT_CACHE_PATH,
    metadata=EMPLOYMENT_CACHE_METADATA,
    expected_pairs=_employment_cache_pairs,
)
_RUN_EMPLOYMENT = _cached_fte is None
if _RUN_EMPLOYMENT:
    # Employment data load from Eurostat.
    emp_eu_bundle = load_emp_eurostat_bundle(MRIO_REGIONS, year=YEAR)
    emp_eu_raw = emp_eu_bundle["employment"].copy()
    hour_eu_raw = emp_eu_bundle["hours"].copy()

    eurostat_emp_inv = assess_industrial_resolution_inventory(
        emp_eu_raw, target_industries=A64_IND, source_db="Eurostat"
    )
    eurostat_hours_inv = assess_industrial_resolution_inventory(
        hour_eu_raw, target_industries=A64_IND, source_db="Eurostat"
    )
    eurostat_countries = (
        set(emp_eu_raw.loc[emp_eu_raw["OBS_VALUE"].notna(), "geo"])
        | set(hour_eu_raw.loc[hour_eu_raw["OBS_VALUE"].notna(), "geo"])
    )
    _source_audit_tables = {
        "Eurostat — Annual total hours": eurostat_hours_inv,
        "Eurostat — Industrial employment": eurostat_emp_inv,
    }
else:
    fte_industry, fte_audit = _cached_fte
    try:
        _source_audit_payload = pd.read_pickle(EMPLOYMENT_AUDIT_CACHE_PATH)
        _source_audit_tables = (
            _source_audit_payload.get("tables", {})
            if isinstance(_source_audit_payload, dict)
            and _source_audit_payload.get("metadata") == EMPLOYMENT_CACHE_METADATA
            else {}
        )
    except Exception:
        _source_audit_tables = {}

def _display_source_audit(table, caption, *, source_profile=False):
    columns = [
        "Country", "Year", "Classification",
        *(["Source profile"] if source_profile else []),
        "Valid source industries", "Resolution",
    ]
    if not isinstance(table, pd.DataFrame):
        print(f"{caption}: no compatible cached inventory is available.")
        return
    shown = table.reindex(columns=columns).copy()
    for field in ("Country", "Year", "Classification"):
        text = shown[field].astype("string").str.strip()
        shown = shown.loc[
            text.notna() & ~text.str.lower().isin({"", "nan", "none"})
        ]
    shown["Valid source industries"] = pd.to_numeric(
        shown["Valid source industries"], errors="coerce"
    )
    shown["Resolution"] = pd.to_numeric(shown["Resolution"], errors="coerce")
    shown = shown.loc[
        shown["Valid source industries"].gt(0) & shown["Resolution"].gt(0)
    ].dropna(subset=["Country", "Year", "Classification"])
    if source_profile:
        shown = shown.dropna(subset=["Source profile"])
        shown = shown.loc[
            ~shown["Source profile"].astype(str).str.contains(
                "unavailable|nan|none", case=False, na=False
            )
        ]
    display(
        shown.style.set_caption(caption).hide(axis="index")
        .set_table_attributes(
            'style="display:block; overflow-x:auto; width:100%; white-space:nowrap;"'
        )
        .set_properties(**{"white-space": "nowrap"})
    )

_display_source_audit(
    _source_audit_tables.get("Eurostat — Annual total hours"),
    "Eurostat — Annual total hours",
)
_display_source_audit(
    _source_audit_tables.get("Eurostat — Industrial employment"),
    "Eurostat — Industrial employment",
)

In [ ]:
if _RUN_EMPLOYMENT:
    oecd_c2_audit = audit_oecd_c2_sources(
        MRIO_REGIONS, EU27, target_industries=A64_IND,
    )
    _source_audit_tables.update(oecd_c2_audit["tables"])

_display_source_audit(
    _source_audit_tables.get("OECD — Annual total hours"),
    "OECD — Annual total hours",
)
_display_source_audit(
    _source_audit_tables.get("OECD — Industrial employment"),
    "OECD — Industrial employment",
)

ILOSTAT


In [ ]:
if _RUN_EMPLOYMENT:
    ilostat_countries = sorted((set(MRIO_REGIONS) - eurostat_countries) | {"X01"})

In [ ]:
# ==============================================================
# Load ILO FTE40 national data for all MRIO regions
# ==============================================================
if _RUN_EMPLOYMENT: fte40_ilo_national = load_ilo_fte40(
    MRIO_REGIONS,
    year=YEAR,
    year_min=YEAR_MIN,
    dataflow_id=FTE40_DATAFLOW,
    basis=FTE40_BASIS,
)

In [ ]:
if _RUN_EMPLOYMENT:
    hour_ilo_raw = load_ilo_mean_hours_by_industry(
        ilostat_countries, year=YEAR, year_min=YEAR_MIN,
        flows=tuple(HOUR_DATAFLOWS), target_industries=A64_IND,
        include_world=True,
    )
    _hours_selection = hour_ilo_raw.attrs.get("selection_log", pd.DataFrame())
    _year_by_iso3 = (
        dict(zip(_hours_selection["geo"], _hours_selection["year"]))
        if not _hours_selection.empty and {"geo", "year"}.issubset(_hours_selection.columns)
        else {}
    )
    emp_ilo_raw = load_ilo_employment_by_industry(
        ilostat_countries, year=YEAR, year_min=YEAR_MIN,
        year_by_iso3=_year_by_iso3, hours_mean=hour_ilo_raw,
        flows=tuple(EMP_DATAFLOWS), target_industries=A64_IND,
        include_world=True,
    )
    ilostat_hours_inv = assess_industrial_resolution_inventory(
        hour_ilo_raw, target_industries=A64_IND, source_db="ILOSTAT"
    )
    ilostat_emp_inv = assess_industrial_resolution_inventory(
        emp_ilo_raw, target_industries=A64_IND, source_db="ILOSTAT"
    )
    _source_audit_tables.update({
        "ILOSTAT — Mean weekly hours": ilostat_hours_inv,
        "ILOSTAT — Industrial employment": ilostat_emp_inv,
    })
    EMPLOYMENT_AUDIT_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    pd.to_pickle(
        {"metadata": EMPLOYMENT_CACHE_METADATA, "tables": _source_audit_tables},
        EMPLOYMENT_AUDIT_CACHE_PATH,
    )

_display_source_audit(
    _source_audit_tables.get("ILOSTAT — Mean weekly hours"),
    "ILOSTAT — Mean weekly hours",
    source_profile=True,
)
_display_source_audit(
    _source_audit_tables.get("ILOSTAT — Industrial employment"),
    "ILOSTAT — Industrial employment",
    source_profile=True,
)

In [ ]:
if _RUN_EMPLOYMENT:
    # Independent, year-locked inputs for the World-minus-FIGARO ROW residual.
    emp_ilo_row_inputs = load_ilo_employment_by_industry(
        MRIO_REGIONS, year=YEAR, year_min=YEAR_MIN,
        flows=tuple(EMP_DATAFLOWS), target_industries=A64_IND,
    )
    emp_ilo_world = load_ilo_global_employment(
        YEAR, year_min=YEAR_MIN, dataflow_id=WORLD_EMPLOYMENT_DATAFLOW,
    )

In [ ]:
if _RUN_EMPLOYMENT:
    _oecd_production_inputs = apply_oecd_c2_production_overrides(
        eurostat_employment=emp_eu_raw,
        ilostat_employment=emp_ilo_raw.loc[~emp_ilo_raw["geo"].astype(str).str.upper().eq("X01")].copy(),
        eurostat_hours=hour_eu_raw,
        oecd_c2_audit=oecd_c2_audit,
    )
    emp_eu_raw = _oecd_production_inputs["eurostat_employment"]
    emp_ilo_raw = _oecd_production_inputs["ilostat_employment"]
    hour_eu_raw = _oecd_production_inputs["eurostat_hours"]

    emp_industry_selected, employment_source_inventory = prepare_fte40_employment_inputs(
        emp_eu_raw,
        emp_ilo_raw.loc[~emp_ilo_raw["geo"].astype(str).str.upper().eq("X01")].copy(),
        target_industries=A64_IND,
    )
    emp_industry_selected.attrs["source_availability_audit"] = employment_source_inventory.copy()
    fte40_source_decision_report = build_fte40_source_decision_report(
        MRIO_REGIONS,
        employment_industry=emp_industry_selected,
        hours_total=hour_eu_raw,
        hours_mean=hour_ilo_raw,
        target_industries=A64_IND,
    )
else:
    fte40_source_decision_report = fte_audit.attrs.get("source_decision_report", pd.DataFrame())
    if fte40_source_decision_report.empty:
        raise RuntimeError("Valid FTE40 cache lacks source-decision provenance; OECD overrides cannot be claimed.")

In [ ]:
if _RUN_EMPLOYMENT:
    VA_BASIC_MRIO = extract_gva_basic(MRIO)
    gva_fte = VA_BASIC_MRIO.rename("value").reset_index()
    gva_fte = gva_fte.rename(
        columns={"destination_region": "geo", "prd_use": "industry"}
    )
    fte_industry, fte_audit = build_fte40_country_industry(
        fte40_ilo_national, mrio_prd_set, hours_total=hour_eu_raw,
        employment_industry=emp_industry_selected, hours_mean=hour_ilo_raw,
        gva=gva_fte, figaro_output=X_MRIO, global_employment=emp_ilo_world,
        row_employment_industry=emp_ilo_row_inputs,
        source_availability_audit=emp_industry_selected.attrs["source_availability_audit"],
        explicit_geos=MRIO_REGIONS, world_geo="X01", row_geo="ROW",
    )
    world_fte_blocks = build_world_fte40_blocks(
        fte40_ilo_national, emp_ilo_world, target_industries=A64_IND,
    )
    fte_industry, fte_audit, _ = apply_world_a14_row_residual(
        fte_industry, fte_audit, world_fte_blocks, target_industries=A64_IND,
    )
    _employment_inventory = employment_source_inventory.copy()
    _oecd_hours_inventory = oecd_c2_audit["audit"]["inventory"].copy()
    _oecd_hours_inventory = _oecd_hours_inventory.loc[
        _oecd_hours_inventory["indicator"].eq("hours")
    ] if not _oecd_hours_inventory.empty else pd.DataFrame()
    _hours_inventory = hour_ilo_raw.attrs.get("source_availability_audit", pd.DataFrame()).copy()
    if not _employment_inventory.empty:
        _employment_inventory["measure"] = "ILOSTAT employment"
        if "source" in _employment_inventory:
            _employment_inventory.loc[_employment_inventory["source"].eq("OECD"), "measure"] = "OECD employment"
    if not _hours_inventory.empty:
        _hours_inventory["measure"] = "ILOSTAT mean weekly hours"
    if not _oecd_hours_inventory.empty:
        _oecd_hours_inventory["measure"] = "OECD annual total hours"
    fte_audit.attrs["source_availability_audit"] = pd.concat(
        [_employment_inventory, _oecd_hours_inventory, _hours_inventory], ignore_index=True, sort=False
    )
    fte_audit.attrs["source_decision_report"] = fte40_source_decision_report.copy()
    EMPLOYMENT_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    _employment_cache_tmp = EMPLOYMENT_CACHE_PATH.with_suffix(".tmp")
    pd.to_pickle(
        {"metadata": EMPLOYMENT_CACHE_METADATA, "fte_industry": fte_industry, "fte_audit": fte_audit},
        _employment_cache_tmp,
    )
    _employment_cache_tmp.replace(EMPLOYMENT_CACHE_PATH)

    # Save supplementary material for C.2 (wages calculation) using the FTE40 results.
    C2_OUTPUT = RESULTS_SUPPLEMENTARY / "Supplementary_Material_C2_Wages_Calculation_FTE40_validation.xlsx"

    export_wage_supplementary_material(
        wage_audit=None,
        emp_core=emp_industry_selected,
        output_path=C2_OUTPUT,
        fte_industry=fte_industry,
        fte_audit=fte_audit,
        hours_total=hour_eu_raw,
        hours_mean=hour_ilo_raw,
    )

In [ ]:
_allocation_summary = build_fte40_allocation_method_comparison(
    None, fte_industry, explicit_geos=MRIO_REGIONS, target_industries=A64_IND,
)["global"][["Allocation method", "After: country-industries", "After: %"]].copy()
_allocation_summary["Allocation method"] = [
    "Annual hours", "Employment × mean weekly hours", "Industrial employment", "FIGARO output",
]
_allocation_summary = _allocation_summary.rename(columns={
    "After: country-industries": "Country-industries", "After: %": "Share (%)",
})
if (
    _allocation_summary["Country-industries"].sum() != len(MRIO_REGIONS) * len(A64_IND)
    or not np.isclose(_allocation_summary["Share (%)"].sum(), 100.0)
):
    raise AssertionError("FTE40 allocation summary is incomplete.")
display(_allocation_summary.style.hide(axis="index"))

# Preserve the downstream W_MRIO interface, now as FTE per million EUR.
W_MRIO = (
    fte_industry.set_index(["geo", "industry"])["FTE_intensity"]
    .rename_axis(["destination_region", "prd_use"])
    .reindex(L1_MRIO.columns)
)
_missing_fte = W_MRIO.isna()
_zero_output = X_MRIO.eq(0)
if (_missing_fte & ~_zero_output).any():
    raise ValueError("FTE coefficients are missing where FIGARO output is positive.")
W_MRIO = W_MRIO.fillna(0.0)

In [ ]:
# C.2.1: Labor compensation per FTE40 by region
fte_mrio = (
    fte_industry.set_index(["geo", "industry"])["FTE_country_industry"]
    .rename_axis(["destination_region", "prd_use"])
    .reindex(L1_MRIO.columns)
)

# Compensation of employees (D1), expressed in EUR per FTE40
output_fte = (
    1_000.0 * (X_MRIO)
    .divide(fte_mrio.where(fte_mrio > 0))
)

gva_per_fte = (
    1_000.0 * VA_BASIC_MRIO / fte_mrio.where(fte_mrio > 0)
)


In [ ]:
plot_wages_dispersion_boxplot_sorted(
    output_fte,
    group_sets={
        "EU27 + EFTA": EU_EFTA,
        "ROE": EUROPE_EXTRA,
    },
    emp_mrio=fte_mrio,
    eu27_countries=EU27,
    stat="median",
    filename="C.2.1.1.pdf",
    save_figures=SAVE_FIGURES,
    close_figures=CLOSE_FIGURES,
    ylabel="Output (kEUR/FTE)",
    show_eu27_median_line=False,
)

In [ ]:
plot_wages_dispersion_boxplot_sorted(
    gva_per_fte,
    group_sets={
        "EU27 + EFTA": EU_EFTA,
        "ROE": EUROPE_EXTRA,
    },
    emp_mrio=fte_mrio,
    eu27_countries=EU27,
    stat="median",
    filename="C.2.1.2.pdf",
    save_figures=SAVE_FIGURES,
    close_figures=CLOSE_FIGURES,
    ylabel="GVA (kEUR/FTE)",
    show_eu27_median_line=True,
)

In [ ]:
print("industries:", fte_industry["industry"].nunique())
print("countries:", fte_industry["geo"].nunique())

In [ ]:
# ============================================================================
# LOW INCOME COUNTRIES
# ============================================================================
# Defined as those with a median wage below EU27 median wage in 2024.

LOW_INCOME = {
    "RU","RS","TR", "MK","ME","AL","JP","CN","AR","BR","ZA",
    "MX","ID","ROW","IN",
}

Model computations


In [ ]:
# FTE/output coefficients are already aligned to the Leontief columns.
covered = W_MRIO.notna()
dest_regions = W_MRIO.index.get_level_values("destination_region").astype(str)
row_cols = (dest_regions == "ROW")
print(
    "FTE coefficient coverage:",
    f"{int(covered.sum())}/{len(W_MRIO)} total;",
    f"ROW {int((covered & row_cols).sum())}/{int(row_cols.sum())}"
)
if row_cols.any() and not W_MRIO.loc[row_cols].notna().any():
    raise ValueError("MRIO has ROW columns, but FTE coefficients have no aligned ROW values.")
W_MRIO = W_MRIO.fillna(0.0)
W_MRIO_CACHE_KEY = f"fte40_{int(pd.util.hash_pandas_object(W_MRIO, index=True).sum()) & 0xFFFFFFFF:08x}"
print("FTE coefficient cache key:", W_MRIO_CACHE_KEY)

In [ ]:
# C.2.2.1: Gross output per FTE40 by industry
plot_lb_box_all_industries(
    output_fte,
    industries=sorted(fte_industry["industry"].unique()),
    europe_group=EUROPE_GROUP,
    filename="C.2.2.1.pdf",
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
    value_scale=1_000.0,
    ylabel="Output (kEUR/FTE)",
)

# C.2.2.2: GVA per FTE40 by industry
plot_lb_box_all_industries(
    gva_per_fte,
    industries=sorted(fte_industry["industry"].unique()),
    europe_group=EUROPE_GROUP,
    filename="C.2.2.2.pdf",
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
    value_scale=1_000.0,
    ylabel="GVA (kEUR/FTE)",
)

In [ ]:
# ==============================================================
# Allocation of MRIO missing origins to rest-of-world (ROW)
# ==============================================================
mrio_origin_set = set(MRIO.index.get_level_values("origin_region").astype(str).unique())
model_origins = set(final_cost_df_agg["origin_country"].unique())
missing_origins = sorted(model_origins - mrio_origin_set)
print("Origins not individually represented in FIGARO:")
print(missing_origins)
display(final_cost_df_agg[final_cost_df_agg["origin_country"].isin(missing_origins)]
    .groupby("origin_country")["value_alloc"]
    .sum()
    .sort_values(ascending=False)
)
final_cost_df_agg["origin_country"] = (
    final_cost_df_agg["origin_country"].astype(str).str.strip()
    .where(lambda s: s.isin(mrio_origin_set), "ROW")
)

### BASELINE RESULTS

In [ ]:
# ============================================================================
# Impact Analysis Module Imports
# ============================================================================

import importlib
import sys

for module_name in [
    "modules.mrio_analysis.mrio",
    "modules.impacts.impacts",
    "modules.impacts.scenarios",
    "modules.cost_model.charts",
    "modules.mrio_analysis.maps",
    "modules.latex_style",
]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from modules.mrio_analysis.mrio import (
    build_y_shock_levels,
    compute_all_results,
    validate_global_mrio_results,
)

from modules.impacts.impacts import (
    build_all_indicators_table_effect_and_comp_columns_from_by_origin,
    compute_industry_effects_by_industry,
    plot_industry_multipliers_twin,
    plot_and_save_eu27_capture_share_figures,
    export_scenario_weight_audit_xlsx,
)

import modules.mrio_analysis.maps as maps
from modules.mrio_analysis.maps import configure_paper_map_figures

from modules.latex_style import (
    df_to_latex,
)

print("✓ Impact Analysis modules imported successfully")

Industrial multipliers


In [ ]:
# ============================================================================
# ============= Code industries selected for the paper analysis ============== 
# ============================================================================
ind_rev = ['B','C24','C25','C27','C28','C33','D35','F',
           'H49','H50','H52','K65','M69_70','M71','M72']

In [ ]:
# ==============================================================
# C.1.2 — Global indirect output multipliers
# ==============================================================

df_by_industry = compute_industry_effects_by_industry(
    ind_rev,
    mrio=MRIO,
    L1_MRIO=L1_MRIO,
    VA_MRIO=VA_MRIO,
    LB_MRIO=LB_MRIO,
    W_MRIO=W_MRIO,
    indicator="Output",
    effect="Indirect",
    shock_eur=1_000_000.0,
    include_deployments=EU27,
)

Plot

In [ ]:
# C.1.2
fig, ax = plot_industry_multipliers_twin(
    df_by_industry,
    indicator="Output",
    effect="Indirect",
    industries=ind_rev,
    ind_labels=ind_labels,
    top_n=5,
    show_xlabel=True,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)


Demand function


In [ ]:
y_shock_lvl, y_shock_sub_lvl = build_y_shock_levels(final_cost_df_agg, ["EU27_AVG"], MRIO)

Baseline calculation

In [ ]:
results_lvl, results_sub_lvl, results_by_origin_lvl, results_by_origin_sub_lvl = compute_all_results(
    y_shock_lvl, y_shock_sub_lvl, YEAR, L1_MRIO, VA_MRIO, LB_MRIO, W_MRIO,
)

validate_global_mrio_results(
    y_shock_lvl[1]["Total"], results_lvl[1]["Total"],
    results_by_origin_lvl[1]["Total"], results_lvl[1],
    L1_MRIO, A_MRIO, VA_MRIO,
)

Table

In [ ]:
REGION_ORDER = ["EU27", "UK", "RoE", "RoW", "Global", "EU27%", "Europe %", "RoW %"]
COMPONENT_ORDER = ["Total", "Devex", "Turbine", "BoP", "Installation", "O&M"]
INDICATORS = ["Output", "GVA", "Jobs"]
# Keep the source capacities before the plotting/table helper overwrites the weighting map.
CONNECTED_COUNTRY_INPUT = {k: dict(v) for k, v in connected_country.items()}
_eu27_power_weights = {
    d: float(CONNECTED_COUNTRY_INPUT[d]["Power (MW)"])
    for d in EU27_DEP
    if d in CONNECTED_COUNTRY_INPUT and float(CONNECTED_COUNTRY_INPUT[d].get("Power (MW)", 0.0)) > 0
}
_eu27_power_weight_sum = sum(_eu27_power_weights.values())
EU27_AVG_CONNECTED_POWER_MW = float(
    TurbineDesign().n_turbines * TurbineDesign().rated_power
)
connected_country = {"EU27_AVG": {"Power (MW)": 1.0}}

lvl = 2
ANNUALIZE = False

NOTE = (
    r"\textit{Note:} GVA is reported in EUR million and employment in "
    r"thousand FTE-years. RoE denotes the rest of Europe, "
    r"excluding the EU27 and the United Kingdom; RoW denotes the rest of the world. "
    r"China and Others are subdivisions of RoW; Others excludes China."
)

# =========================
# EFFECT + COMPONENT COLUMNS
# Direct / Indirect + Total by component
# =========================
effect_comp_cols = build_all_indicators_table_effect_and_comp_columns_from_by_origin(
    results_by_origin_sub_lvl,
    core=[EU27_AVG],
    connected_country=connected_country,
    indicators=INDICATORS,
    metric_order=REGION_ORDER,
    region_order=REGION_ORDER,
    annualize=ANNUALIZE,
    lifetime_years=lifetime_years,
)

summary_path = Path(TAB_ROOT) / "summary_indicators_compcols_total.tex"

latex = df_to_latex(
    effect_comp_cols,
    caption="Cumulative 25-year economic impacts of a 1 GW EU-27 OWF by effect type and component.",
    label="tab:summary_indicators_compcols_total",
    note=NOTE,
)

summary_path.write_text(latex, encoding="utf-8")

print("OK combined effect/component table saved:")
print(project_relative_path(summary_path))

Plots by component and subcomponent

In [ ]:
SPLIT_COUNTRIES = ('GB','NO','TR','CN','RS',"IN",'KR','US','ROW')

capture_share_figures = plot_and_save_eu27_capture_share_figures(
    results_by_origin_sub_lvl=results_by_origin_sub_lvl,
    connected_country=connected_country,
    split_origin_countries=SPLIT_COUNTRIES,
    avg_dep_order=(EU27_AVG,),
    year=YEAR,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

Geographical distribution


In [ ]:
components_map = {
    "Turbine": ["Turbine"],
    "Devex & O&M": ["Devex", "O&M"],
    "Balance of plant": ["BoP"],
    "Installation": ["Installation"],
}

EU maps

In [ ]:
# D.1 EU27 maps
maps.plot_paper_eu27_component_share_maps(
    results_by_origin=results_by_origin_lvl,
    components_map=components_map,
    core_deployment=[EU27_AVG],
    eu27=EU27,
    connected_country=connected_country,
    lifetime_years=lifetime_years,
    save=SAVE_MAPS,
    close=CLOSE_FIGURES,
)

Global maps

In [ ]:
# D.2 global maps
maps.plot_paper_world_component_share_maps(
    results_by_origin=results_by_origin_lvl,
    components_map=components_map,
    core_deployment=[EU27_AVG],
    eu27=EU27,
    connected_country=connected_country,
    save=SAVE_MAPS,
    close=CLOSE_FIGURES,
)

### OUTSOURCING/RESILIENCE SCENARIOS


In [ ]:
import importlib
import sys

import modules.impacts.scenarios as scenarios
scenarios = importlib.reload(scenarios)

from modules.impacts.scenarios import (
    maybe_run_or_load,
    maybe_run_eu27_side_from_net,
    build_baseline_panels,
    build_eu27_procurement_savings_df,
    prepare_resilience_markup_sensitivities,
    plot_and_save_eu27_counterfactual_figures,
    build_eu27_resilience_accounting_table,
    build_eu27_net_delta_effect_df,
    build_eu27_component_tradeoff_curve_data,
    build_eu27_aggregate_tradeoff_curve_data,
    eu27_component_tradeoff_axis_limits,
    plot_eu27_component_cost_impact_curves,
    plot_eu27_aggregate_cost_impact_curves,
)


In [ ]:
# =============================
# Scenario inputs
# =============================

scope = "EU27"
SCENARIO_CACHE_V10 = RESULTS_CACHE / "v10_quantity_recalibrated_mu"
exclude_deployments_resilience = ()

SCENARIO_RECOMPUTE = RECOMPUTE
EU_SIDE_RECOMPUTE = RECOMPUTE
EU27_PANEL_DEPLOYMENTS = list(EU27) + [EU27_AVG]

DELTA_BASE = 0
DELTA_RESIL = 100
DELTA_OUT = -100
DELTAS = sorted(set(range(DELTA_OUT, DELTA_RESIL + 1, 5)))

# Markup assumptions for the main scenario figures
MARKUP_EU_LOW = 0.10
MARKUP_EU_CENTRAL = 0.30
MARKUP_EU_HIGH = 0.50
MARKUP_EU = MARKUP_EU_CENTRAL

# Markup sensitivities included in the resilience accounting table
ACCOUNTING_MARKUPS = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6)

# Independent control for accounting-only sensitivity recomputation
ACCOUNTING_RECOMPUTE = RECOMPUTE

# Cost adjustment sensitivity bounds are retained for the resilience
# accounting table. Figure 3.4 and Figure 3.5 use only the central markup.
# Source: Oxford Institute for Energy Studies, OIES Paper CE17 (2025).
# https://www.oxfordenergy.org/wpcms/wp-content/uploads/2025/10/
# CE17-Chinese-participation-in-Europes-offshore-wind-sector.pdf

# lvl_comp=2 supplies the broad Turbine and BoP manufacturing groups;
# lvl_sim=0 retains the detailed capacity resolution used by the model.
lvl_comp = 2
lvl_sim = 0

# RoW countries shown separately in the D.3-derived Figure 3.4 panels.
SPLIT_ROW = ["GB", "NO", "CN", "US", "TR", "RU", "KR", "JP", "IN"]

TARGET_COMPONENT_FILTER_TURBINE = {"Turbine"}
TARGET_COMPONENT_FILTER_FARM = {"BoP"}


#### Baseline scenario

In [ ]:
# =============================
# Load/run central EU27 manufacturing sourcing scenarios (mu = 0.30)
# =============================
sd_base = {}
sr_base = {}

sd_base[scope], sr_base[scope] = maybe_run_or_load(
    cache_dir="EU27_resilience_Turbine_BoP",
    cache_base_dir=SCENARIO_CACHE_V10,
    force_recompute=SCENARIO_RECOMPUTE,
    execution_mode=("force_recompute" if SCENARIO_RECOMPUTE else "compute_missing"),
    require_detailed=not SCENARIO_RECOMPUTE,
    EUROPE=EU27,
    EU27=EU27,
    LOW_INCOME=LOW_INCOME,
    df_alloc=df_alloc_eu27,
    flows_by_component_hier_all=flows_eu27_avg,
    ROOT=str(HELPERS_DIR),
    deltas_pp=DELTAS,
    L1_MRIO=L1_MRIO,
    VA_MRIO=VA_MRIO,
    LB_MRIO=LB_MRIO,
    W_MRIO=W_MRIO,
    receiver_policy="baseline_mix",
    results_by_origin_lvl=results_by_origin_lvl,
    results_by_origin_sub_lvl=results_by_origin_sub_lvl,
    capacity_lvl=lvl_sim,
    connected_country=None,
    eu_spend_reduction_pct=MARKUP_EU,
    preserve_total=False,
    component_filter_turbine=TARGET_COMPONENT_FILTER_TURBINE,
    component_filter_farm=TARGET_COMPONENT_FILTER_FARM,
    mrio=MRIO,
    core=[EU27_AVG],
    annualize=ANNUALIZE,
    lifetime_years=lifetime_years,
)


#### EU27-side decomposition

In [ ]:
# =============================
# Load/run EU27-side decomposition for the central scenario
# =============================
sd_eu_side = {}
sr_eu_side = {}

sd_eu_side[scope], sr_eu_side[scope] = maybe_run_eu27_side_from_net(
    net_scenarios_delta=sd_base[scope],
    net_results_delta=sr_base[scope],
    eu_countries=EU27,
    cache_dir=f"{scope}_eu27_side_macro_Turbine_BoP_central",
    cache_base_dir=SCENARIO_CACHE_V10,
    force_recompute=EU_SIDE_RECOMPUTE,
    execution_mode=("force_recompute" if EU_SIDE_RECOMPUTE else "compute_missing"),
    deltas_pp=DELTAS,
    L1_MRIO=L1_MRIO,
    VA_MRIO=VA_MRIO,
    LB_MRIO=LB_MRIO,
    W_MRIO=W_MRIO,
    mrio=MRIO,
    core=[EU27_AVG],
    annualize=ANNUALIZE,
    lifetime_years=lifetime_years,
)


#### Manufacturing counterfactuals

In [ ]:
# =============================
# Figure 3.4 Manufacturing-only counterfactuals
# Total is constructed from Turbine + BoP only.
# =============================
_, PANEL_34_D3_ALL = build_baseline_panels(
    sr_base[scope],
    split_row=SPLIT_ROW,
    EU27=EU27_PANEL_DEPLOYMENTS,
    connected_country=None,
    exclude_deployments=exclude_deployments_resilience,
    add_eu_indirect_split=True,
    lvl_plot=lvl_comp,
)
PANEL_34_D3 = PANEL_34_D3_ALL.loc[
    PANEL_34_D3_ALL["Component"].astype(str).isin({"Turbine", "BoP"})
].copy()
if set(PANEL_34_D3["Component"].astype(str).unique()) != {"Turbine", "BoP"}:
    raise ValueError("Figure 3.4 requires both manufacturing groups: Turbine and BoP.")
_34_abs_cols = [c for c in PANEL_34_D3.columns if c.startswith("abs_")]
_34_total = PANEL_34_D3.groupby(
    ["delta_pp", "scenario", "Region"], as_index=False
)[_34_abs_cols].sum()
_34_total["Component"] = "Total"
PANEL_COMP_D3 = pd.concat([PANEL_34_D3, _34_total], ignore_index=True, sort=False)

# Keep complete project panels for the accounting table. Figure 3.4 receives
# manufacturing-only copies so unchanged Devex, Installation, and O&M levels
# cannot enter its absolute Total.
_, PANEL_COMP_NET_RAW = build_baseline_panels(
    sr_base[scope], EU27=EU27, connected_country=None,
    exclude_deployments=exclude_deployments_resilience, lvl_plot=lvl_comp,
)
PANEL_COMP_NET_RAW = PANEL_COMP_NET_RAW.loc[
    PANEL_COMP_NET_RAW["scenario"].astype(str).eq("baseline")
].copy()
_, PANEL_COMP_EU_SIDE_RAW = build_baseline_panels(
    sr_eu_side[scope], EU27=EU27, connected_country=None,
    exclude_deployments=exclude_deployments_resilience, lvl_plot=lvl_comp,
)
PANEL_COMP_EU_SIDE_RAW = PANEL_COMP_EU_SIDE_RAW.loc[
    PANEL_COMP_EU_SIDE_RAW["scenario"].astype(str).eq("baseline")
].copy()
_manufacturing = {"Turbine", "BoP"}
PANEL_COMP_NET = PANEL_COMP_NET_RAW.loc[
    PANEL_COMP_NET_RAW["Component"].astype(str).isin(_manufacturing)
].copy()
PANEL_COMP_EU_SIDE = PANEL_COMP_EU_SIDE_RAW.loc[
    PANEL_COMP_EU_SIDE_RAW["Component"].astype(str).isin(_manufacturing)
].copy()

EU27_COUNTERFACTUAL_PATHS = plot_and_save_eu27_counterfactual_figures(
    PANEL_D3=PANEL_COMP_D3,
    PANEL_NET=PANEL_COMP_NET,
    PANEL_EU_SIDE=PANEL_COMP_EU_SIDE,
    split_row=SPLIT_ROW,
    save=SAVE_FIGURES_RES,
    close=CLOSE_FIGURES,
)


#### Component-level sourcing trade-offs

In [ ]:
# =============================
# Markup sensitivity preparation
# =============================
_markup_prepared = prepare_resilience_markup_sensitivities(
    central_scenarios=(sd_base[scope], sr_base[scope]),
    accounting_markups=ACCOUNTING_MARKUPS,
    cache_base_dir=SCENARIO_CACHE_V10,
    figure_deltas=DELTAS,
    scenario_recompute=SCENARIO_RECOMPUTE,
    accounting_recompute=ACCOUNTING_RECOMPUTE,
    EUROPE=EU27,
    EU27=EU27,
    LOW_INCOME=LOW_INCOME,
    df_alloc=df_alloc_eu27,
    flows_by_component_hier_all=flows_eu27_avg,
    ROOT=str(HELPERS_DIR),
    L1_MRIO=L1_MRIO,
    VA_MRIO=VA_MRIO,
    LB_MRIO=LB_MRIO,
    W_MRIO=W_MRIO,
    mrio=MRIO,
    core=[EU27_AVG],
    annualize=ANNUALIZE,
    lifetime_years=lifetime_years,
    receiver_policy="baseline_mix",
    results_by_origin_lvl=results_by_origin_lvl,
    results_by_origin_sub_lvl=results_by_origin_sub_lvl,
    capacity_lvl=lvl_sim,
    connected_country=None,
    component_filter_turbine=TARGET_COMPONENT_FILTER_TURBINE,
    component_filter_farm=TARGET_COMPONENT_FILTER_FARM,
    preserve_total=False,
    exclude_deployments=exclude_deployments_resilience,
    figure_markup_low=MARKUP_EU_LOW,
    figure_markup_central=MARKUP_EU_CENTRAL,
    figure_markup_high=MARKUP_EU_HIGH,
    delta_base=DELTA_BASE,
    delta_out=DELTA_OUT,
    delta_resil=DELTA_RESIL,
    accounting_component_level=lvl_comp,
)

sd_markup = _markup_prepared["sd_markup"]
sr_markup = _markup_prepared["sr_markup"]
MARKUP_NET_EFFECTS = _markup_prepared["markup_net_effects"]
PROCUREMENT_SAVINGS_34 = _markup_prepared["procurement_savings"]
sd_accounting = _markup_prepared["sd_accounting"]
sr_accounting = _markup_prepared["sr_accounting"]
MARKUP_NET_EFFECTS_ACCOUNTING = _markup_prepared["markup_net_effects_accounting"]
PROCUREMENT_SAVINGS_ACCOUNTING = _markup_prepared["procurement_savings_accounting"]
ACCOUNTING_NOTE = _markup_prepared["accounting_note"]

In [ ]:
# =============================
# Figure 3.5 Component-level sourcing trade-offs (mu = 0.30)
# =============================
FIGURE_35_TRADEOFF_NORMALIZATION = "absolute"
# Reload explicitly here so rerunning Figure 3.5 in an existing kernel
# cannot retain stale function aliases from an earlier module version.
import importlib
import modules.impacts.scenarios as scenarios
scenarios = importlib.reload(scenarios)

EU27_COMPONENT_FIGURE_PATHS = {}
EU27_COMPONENT_CURVE_DATA = scenarios.build_eu27_component_tradeoff_curve_data(
    scenarios_delta_by_markup={
        MARKUP_EU_LOW: sd_markup["low"],
        MARKUP_EU_CENTRAL: sd_markup["central"],
        MARKUP_EU_HIGH: sd_markup["high"],
    },
    scenarios_result_by_markup={
        MARKUP_EU_LOW: sr_markup["low"],
        MARKUP_EU_CENTRAL: sr_markup["central"],
        MARKUP_EU_HIGH: sr_markup["high"],
    },
    expected_markups=(MARKUP_EU_LOW, MARKUP_EU_CENTRAL, MARKUP_EU_HIGH),
    expected_deltas=tuple(DELTAS),
    annualize=ANNUALIZE,
    low_income=LOW_INCOME,
)
EU27_COMPONENT_AXIS_LIMITS = {
    (group, indicator): scenarios.eu27_component_tradeoff_axis_limits(
        EU27_COMPONENT_CURVE_DATA,
        indicator=indicator,
        baseline_markup=MARKUP_EU_CENTRAL,
        padding=0.04,
        normalization_scope=FIGURE_35_TRADEOFF_NORMALIZATION,
        component_group=group,
    )
    for group in ("Turbine", "BoP")
    for indicator in ("GVA", "Jobs")
}
for group in ("Turbine", "BoP"):
    for indicator in ("GVA", "Jobs"):
        indicator_file = "GVA" if indicator == "GVA" else "Employment"
        filename = f"3.5_eu27_sourcing_tradeoff_{indicator_file}_{group}.pdf"
        _, _, output_path = scenarios.plot_eu27_component_cost_impact_curves(
            EU27_COMPONENT_CURVE_DATA,
            component_group=group,
            indicator=indicator,
            filename=filename,
            baseline_markup=MARKUP_EU_CENTRAL,
            axis_limits=EU27_COMPONENT_AXIS_LIMITS[(group, indicator)],
            normalization_scope=FIGURE_35_TRADEOFF_NORMALIZATION,
            annualize=ANNUALIZE,
            save=SAVE_FIGURES_RES,
            close=CLOSE_FIGURES,
        )
        EU27_COMPONENT_FIGURE_PATHS[f"curve_{group}_{indicator}"] = output_path

EU27_AGGREGATE_CURVE_DATA = scenarios.build_eu27_aggregate_tradeoff_curve_data(
    EU27_COMPONENT_CURVE_DATA,
    scenarios_delta_by_markup={
        MARKUP_EU_LOW: sd_markup["low"],
        MARKUP_EU_CENTRAL: sd_markup["central"],
        MARKUP_EU_HIGH: sd_markup["high"],
    },
    scenarios_result_by_markup={
        MARKUP_EU_LOW: sr_markup["low"],
        MARKUP_EU_CENTRAL: sr_markup["central"],
        MARKUP_EU_HIGH: sr_markup["high"],
    },
    low_income=LOW_INCOME,
)
EU27_AGGREGATE_AXIS_LIMITS = {
    indicator: scenarios.eu27_component_tradeoff_axis_limits(
        EU27_AGGREGATE_CURVE_DATA,
        indicator=indicator,
        baseline_markup=MARKUP_EU_CENTRAL,
        padding=0.04,
        normalization_scope=FIGURE_35_TRADEOFF_NORMALIZATION,
        required_series=("Turbine", "BoP", "Total project"),
    )
    for indicator in ("GVA", "Jobs")
}
EU27_AGGREGATE_FIGURE_PATHS = {}
for indicator in ("GVA", "Jobs"):
    indicator_file = "GVA" if indicator == "GVA" else "Employment"
    filename = f"3.5_eu27_sourcing_tradeoff_{indicator_file}_Aggregated.pdf"
    _, _, output_path = scenarios.plot_eu27_aggregate_cost_impact_curves(
        EU27_AGGREGATE_CURVE_DATA,
        indicator=indicator,
        filename=filename,
        baseline_markup=MARKUP_EU_CENTRAL,
        axis_limits=EU27_AGGREGATE_AXIS_LIMITS[indicator],
        normalization_scope=FIGURE_35_TRADEOFF_NORMALIZATION,
        annualize=ANNUALIZE,
        save=SAVE_FIGURES_RES,
        close=CLOSE_FIGURES,
    )
    EU27_AGGREGATE_FIGURE_PATHS[f"curve_{indicator}"] = output_path


Accounting table

In [ ]:
# =============================
# Scenarios accounting table
# =============================
summary = build_eu27_resilience_accounting_table(
    PANEL_NET=PANEL_COMP_NET_RAW,
    PANEL_EU_SIDE=PANEL_COMP_EU_SIDE_RAW,
    delta_base=DELTA_BASE,
    delta_resil=[DELTA_RESIL, DELTA_OUT],
    component_order=["Total", "Turbine", "BoP"],
    include_scenario_row=True,
    procurement_savings=PROCUREMENT_SAVINGS_ACCOUNTING,
    markup_net_effects=MARKUP_NET_EFFECTS_ACCOUNTING,
)

summary = summary.rename(
    index={
        f"Delta = {DELTA_RESIL}": f"Reshoring ($\\delta={DELTA_RESIL}$)",
        f"Delta = {DELTA_OUT}": f"Offshoring ($\\delta={DELTA_OUT}$)",
    },
    level="Scenario",
)
latex = df_to_latex(
    summary,
    caption="EU27 effects of full resilience and offshoring sourcing scenarios for a 1 GW EU-27 OWF.",
    label="tab:eu27_resilience_accounting",
    note=ACCOUNTING_NOTE,
)
summary_path = Path(TAB_ROOT) / "summary_eu27_resilience_accounting.tex"
summary_path.write_text(latex, encoding="utf-8")
print(f"OK EU27 resilience effects table saved: {project_relative_path(summary_path)}")

In [ ]:
# Export detailed saved Results plus the recorded scenario-weight audit.
SCENARIO_AUDIT_XLSX = export_scenario_weight_audit_xlsx(
    scenarios_delta=sd_base[scope],
    scenarios_result=sr_base[scope],
    scenarios_delta_by_markup={
        MARKUP_EU_LOW: sd_markup["low"],
        MARKUP_EU_CENTRAL: sd_markup["central"],
        MARKUP_EU_HIGH: sd_markup["high"],
    },
    scenarios_result_by_markup={
        MARKUP_EU_LOW: sr_markup["low"],
        MARKUP_EU_CENTRAL: sr_markup["central"],
        MARKUP_EU_HIGH: sr_markup["high"],
    },
    EU27=EU27,
    connected_country=None,
    connected_power_mw=EU27_AVG_CONNECTED_POWER_MW,
    baseline_level=lvl_sim,
    annualize=ANNUALIZE,
    lifetime_years=lifetime_years,
)